<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_37_typing_pydantic/note_lesson_37_pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧾 Урок 37 — Typing + Pydantic: новинний агрегатор, крок 1

З цього уроку FastAPI-гілка курсу будує **новинний агрегатор**. Починаємо з готового парсера rbc.ua `parse_rbc_news` і 168 новин, які він зібрав. Парсер повертає словники — а словник нічого не гарантує. Сьогодні два рефакторинги: **анотації типів** (їх перевіряє `mypy` до запуску) і **Pydantic-модель** `NewsItem` (вона перевіряє дані під час роботи).

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | `news_hub` і знімок 168 новин rbc.ua: що в сирих даних | 0.1 | |
| 1 | анотації: лише підказки, `list[...]`, `X \| None`, `Literal`, `Callable`, `TypedDict`; `mypy` до і після | 1.1–1.6 | **Задача 1** |
| 2 | `NewsItem`: перевірка справжніх новин, усі помилки разом, `frozen`, приведення типів, дані з URL, відхилені новини | 2.1–2.7 | |
| 3 | власні моделі: `Field`, `Literal`, `field_validator`, `model_validator(mode="after")`, вкладені моделі, `computed_field`, `strict`, `extra="forbid"`, `model_dump` | 3.1–3.8 | **Задача 2** |
| 4 | весь знімок, JSON туди й назад, JSON Schema, `TypeAdapter`, парсер на HTML | 4.1–4.7 | |
| 5 | знайди помилку: URL рядком | 5.1 | **Задача 3** |
| 6 | тести і `mypy --strict` проєкту | 6.1 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- `mypy` запускається окремою програмою; перший запуск — до пів хвилини (він читає типи бібліотек), наступні — швидше.
- Теорія й діфи рефакторингу — у книзі курсу: [Урок 37](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_37/).

---
## 0. Проєкт у ноутбуці

`news_hub` — стартовий парсер `parse_rbc_news` з анотаціями типів (`parser.py`), модель `NewsItem` (`models.py`) і знімок стрічки rbc.ua (`data/rbc_news_snapshot.json`, 168 новин). Мережа не потрібна: усе працює на знімку. У Colab клітинка нижче завантажить лише папку уроку і встановить залежності.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_37_typing_pydantic"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if os.path.basename(os.getcwd()) == "news_hub" and os.path.isdir("news_hub"):   # клітинку запустили вдруге
    os.chdir("..")
if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import bs4, mypy, pydantic, pytest  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
print("папка:", os.getcwd().split(os.sep)[-2:])

In [ ]:
import inspect
import json
import tempfile
from collections import Counter
from datetime import datetime, time
from pathlib import Path
from typing import Callable, Literal
from urllib.parse import urlsplit

from pydantic import (BaseModel, ConfigDict, Field, HttpUrl, TypeAdapter, ValidationError, computed_field,
                      field_validator, model_validator)

from news_hub.models import NewsItem, Rejected, validate_news
from news_hub.parser import RawNews, parse_rbc_news
from news_hub.snapshot import load_snapshot

raw_news = load_snapshot()
print("новин у знімку:", len(raw_news))
print(raw_news[1])

### Приклад 0.1. Що насправді в сирих даних

Перш ніж описувати типи, подивимось на дані: які поля порожні, у якому форматі час, з яких доменів адреси.

In [ ]:
print("ключі:", list(raw_news[0]))
print("порожніх category:", sum(not n["category"] for n in raw_news),
      "| description:", sum(not n["description"] for n in raw_news),
      "| datetime:", sum(not n["datetime"] for n in raw_news))
print("довжина datetime:", Counter(len(n["datetime"]) for n in raw_news))
print("домени:", Counter(urlsplit(n["url"]).hostname for n in raw_news))
print("шляхи:", Counter("/".join(urlsplit(n["url"]).path.split("/")[1:3]) for n in raw_news))

- `category` і `description` **порожні завжди** — стрічка їх не містить; категорію й мову доведеться виводити з URL (`/rus/news/…`);
- `datetime` — або `""` (31 новина), або лише час `"HH:MM"` (5 символів) без дати;
- одна новина — з піддомену `auto.rbc.ua`: код, що «ріже» URL рядком, на ній помилиться (розділ 5).

Зі словника `list[dict]` усе це з'ясовується лише під час запуску. Рефакторимо.

---
## 1. Анотації типів

**Анотація** — підказка в коді: `def headline(item: RawNews) -> str`. Python її **не перевіряє**; перевіряє окрема програма `mypy` — до запуску, як лінтер.

| Запис | Що означає | Приклад з агрегатора |
|---|---|---|
| `str`, `int`, `bool` | простий тип | `title: str` |
| `list[str]`, `set[str]` | список, множина рядків | `seen_urls: set[str]` |
| `dict[str, int]` | словник: ключ → значення | `count_by_lang(...) -> dict[str, int]` |
| `X \| None` | або `X`, або `None` | `published_time: time \| None` |
| `Literal["uk", "ru"]` | лише ці значення | `lang` |
| `Callable[[A], B]` | функція: аргумент `A` → результат `B` | фільтр новин |
| `TypedDict` | словник з **відомими** ключами й типами | `RawNews` |

### Приклад 1.1. Анотації — лише підказки

Опишемо функцію з анотаціями й передамо їй **не той** тип.

In [ ]:
def headline(item: RawNews) -> str:
    return item["title"].upper()

In [ ]:
print(headline.__annotations__)
print(headline(raw_news[1])[:40])

fake: RawNews = {"title": 42, "url": None, "category": "", "description": "", "datetime": ""}  # type: ignore
print("змінна з «неправильним» словником:", type(fake).__name__, fake["title"], fake["url"])
try:
    headline(fake)
except AttributeError as error:
    print("впало лише всередині:", error)

Анотації зберігаються в `__annotations__`, але під час роботи ніхто їх не дивиться: `fake` з числом замість рядка спокійно створився й передався у функцію. Впало лише на `.upper()` — у числа немає такого методу. `# type: ignore` у рядку — це для mypy: «тут неправильно навмисно, не скаржся».

### Приклад 1.2. Типи з таблиці в справжніх функціях

Чотири невеликі функції над сирими новинами: список, «знайшлося або `None`», функція як параметр і `Literal`.

In [ ]:
LangCode = Literal["uk", "ru"]          # псевдонім типу: далі пишемо LangCode замість Literal[...]


def titles(items: list[RawNews]) -> list[str]:
    return [item["title"] for item in items]


def first_with(items: list[RawNews], word: str) -> RawNews | None:
    """Перша новина зі словом у заголовку або None, якщо такої немає."""
    for item in items:
        if word.lower() in item["title"].lower():
            return item
    return None


def keep_if(items: list[RawNews], rule: Callable[[RawNews], bool]) -> list[RawNews]:
    return [item for item in items if rule(item)]


def lang_of(url: str) -> LangCode | None:
    first = urlsplit(url).path.split("/")[1]
    return {"ukr": "uk", "rus": "ru"}.get(first)  # type: ignore[return-value]

In [ ]:
print(titles(raw_news[:2]))
found = first_with(raw_news, "бюджет")
print("бюджет:", found["title"] if found is not None else "немає")
print("слово, якого немає:", first_with(raw_news, "абракадабра"))
no_time = keep_if(raw_news, lambda item: item["datetime"] == "")
print("без часу:", len(no_time))
print("мови:", Counter(lang_of(item["url"]) for item in raw_news))

- `RawNews | None` чесно каже: «може й не знайтися». Перш ніж брати `found["title"]`, треба перевірити `is not None` — інакше mypy поскаржиться (приклад 1.5);
- `Callable[[RawNews], bool]` — сюди можна передати будь-яку функцію «новина → так/ні», зокрема `lambda`;
- `.get()` словника повертає `str | None`, а не `Literal` — mypy цього не знає, тому в `lang_of` позначено `# type: ignore[return-value]`. У моделі `NewsItem` той самий словник має тип `dict[str, Literal["uk", "ru"]]` — і обхід не потрібен.

### Приклад 1.3. `count_by_lang` з анотаціями

Функція над **перевіреними** новинами (`NewsItem`, розділ 2): приймає список моделей, повертає словник «мова → кількість». Змінній `counts` теж дамо анотацію: порожній `{}` mypy не знає, чим буде заповнено.

In [ ]:
def count_by_lang(items: list[NewsItem]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for item in items:
        counts[item.lang] = counts.get(item.lang, 0) + 1
    return counts

In [ ]:
annotations = count_by_lang.__annotations__
assert count_by_lang([NewsItem.from_raw(raw_news[0])]) == {"ru": 1}
assert annotations["items"] == list[NewsItem] and annotations["return"] == dict[str, int]
print(annotations)
print(count_by_lang([NewsItem.from_raw(raw) for raw in raw_news]))

`item.lang` — атрибут моделі, і mypy знає, що це `"uk"` або `"ru"`: одруківка `item.lnag` була б помилкою ще до запуску.

### Приклад 1.4. `mypy`: словник проти `TypedDict`

У папці `examples/` — дві версії тих самих функцій з тими самими помилками: «було» зі звичайним `dict` і «стало» з `RawNews` і `NewsItem`. `run_mypy` запускає mypy окремою програмою; спільна тимчасова папка кешу пришвидшує повторні запуски.

In [ ]:
MYPY_CACHE = tempfile.mkdtemp(prefix="mypy_cache_")


def run_mypy(*args):
    """Запустити mypy з папки news_hub (тут лежить пакет news_hub) і показати його вивід."""
    result = subprocess.run([sys.executable, "-m", "mypy", "--no-color-output", "--cache-dir", MYPY_CACHE, *args],
                            capture_output=True, text=True, env={**os.environ, "MYPYPATH": os.getcwd()})
    print(result.stdout.strip() or result.stderr.strip())
    return result

In [ ]:
print(open("examples/after_typed.py", encoding="utf-8").read())
before = run_mypy("examples/before_dict.py")
print()
after = run_mypy("examples/after_typed.py")
assert before.returncode == 0 and after.returncode == 1

- для `dict` будь-який ключ «правильний» — mypy мовчить, `"titel"` чекає на запуск і `KeyError`;
- з `TypedDict` mypy знає ключі й підказує `Did you mean "title"?`;
- `time | None` змушує обробити `None` — випадок, який у реальних даних трапляється 31 раз. Код повернення mypy — `1`: так CI (урок 51) зупиняє збірку.

### Приклад 1.5. Виправляємо код, а не анотації

Ті самі дві функції — у файлі, який пише клітинка. Спершу з помилками, потім виправлені: ключ `"title"` і перевірка на `None`. Анотації не змінюються — змінюється код.

In [ ]:
BROKEN = """
from news_hub.models import NewsItem
from news_hub.parser import RawNews


def headline(item: RawNews) -> str:
    return item["titel"].upper()


def hour(item: NewsItem) -> int | None:
    return item.published_time.hour
"""

FIXED = """
from news_hub.models import NewsItem
from news_hub.parser import RawNews


def headline(item: RawNews) -> str:
    return item["title"].upper()


def hour(item: NewsItem) -> int | None:
    if item.published_time is None:
        return None
    return item.published_time.hour
"""


def mypy_source(source, *flags):
    """Записати код у тимчасовий файл і перевірити його mypy."""
    path = Path(tempfile.mkdtemp()) / "my_check.py"
    path.write_text(source, encoding="utf-8")
    return run_mypy(*flags, str(path))

In [ ]:
broken = mypy_source(BROKEN)
print()
fixed = mypy_source(FIXED)
assert broken.returncode == 1 and fixed.returncode == 0

Після `if item.published_time is None: return None` mypy **звужує** тип: нижче цього рядка `published_time` — уже `time`, а не `time | None`, і `.hour` дозволено. Тип результату `int | None` чесно каже викликачу, що години може не бути.

### Приклад 1.6. `TypedDict` під час роботи — звичайний словник

In [ ]:
item = RawNews(title="Уряд схвалив бюджет", url="https://www.rbc.ua/ukr/news/b.html",
               category="", description="", datetime="09:30")
print(type(item).__name__, item)
print("обов'язкові ключі:", sorted(RawNews.__required_keys__))
print("типи:", {key: tp.__name__ for key, tp in RawNews.__annotations__.items()})

`RawNews(...)` — не новий клас об'єктів, а звичайний `dict`: жодних перевірок, жодних витрат під час роботи. `TypedDict` існує для mypy — і для Pydantic, який уміє перевірити такий словник (приклад 4.5).

### Задача 1. Гістограма годин з анотаціями

Напиши `hours_histogram(items)` **з анотаціями**: приймає `list[NewsItem]`, повертає `dict[int, int]` — «година публікації → скільки новин». Новини без часу (`published_time is None`) пропускай.

Перевірка — у наступній клітинці: вона порівняє результат зі знімком, перевірить анотації і запустить на твоїй функції `mypy --strict`.

In [ ]:
# BEGIN SOLUTION
def hours_histogram(items: list[NewsItem]) -> dict[int, int]:
    histogram: dict[int, int] = {}
    for item in items:
        if item.published_time is None:
            continue
        hour = item.published_time.hour
        histogram[hour] = histogram.get(hour, 0) + 1
    return histogram
# END SOLUTION

In [ ]:
assert "hours_histogram" in globals(), "визнач у клітинці вище функцію hours_histogram з анотаціями"
items = [NewsItem.from_raw(raw) for raw in raw_news]
histogram = hours_histogram(items)
assert isinstance(histogram, dict), "функція має повернути словник «година → кількість»"
expected = Counter(item.published_time.hour for item in items if item.published_time is not None)
assert histogram == dict(expected), "кожна година — скільки новин; новини без часу не рахуємо"
assert sum(histogram.values()) == 137
assert hours_histogram.__annotations__ == {"items": list[NewsItem], "return": dict[int, int]}, "потрібні анотації параметра й результату"
strict = mypy_source("from news_hub.models import NewsItem\n\n\n" + inspect.getsource(hours_histogram), "--strict")
assert strict.returncode == 0, "mypy --strict знаходить помилки"
print(dict(sorted(histogram.items())))
print("✅ Задача 1 розв'язана")

---
## 2. `NewsItem`: перевірка справжніх новин

**Pydantic** перетворює анотації на **перевірку**: клас-модель описує поля й типи, а створення об'єкта або перевіряє й приводить дані, або кидає `ValidationError`.

```python
class NewsItem(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, frozen=True)

    title: str = Field(min_length=10, max_length=300)
    url: HttpUrl
    source: str                      # домен без www — з URL
    lang: Literal["uk", "ru"]        # /ukr/ або /rus/ — з URL
    category: str                    # /ukr/news/ → «Новини» — з URL
    published_time: time | None = None
```

`NewsItem.from_raw(raw)` перекладає сирий `RawNews` у поля моделі; `source`, `lang` і `category` виводить з URL валідатор `derive_from_url`.

### Приклад 2.1. Перша новина знімка

In [ ]:
first = NewsItem.from_raw(raw_news[0])

assert first.title == "Реформа ЗСУ" and first.published_time is None, "у цієї новини datetime порожній"
assert (first.source, first.lang, first.category) == ("rbc.ua", "ru", "Новини")
print(repr(first))

Порожній `datetime` став `None`: валідатор `derive_from_url` відкидає порожні рядки, і поле отримує значення за замовчуванням. `url` — уже не рядок, а `HttpUrl`.

### Приклад 2.2. Модель → словник і JSON

In [ ]:
item = NewsItem.from_raw(raw_news[1])
print(type(item.published_time).__name__, item.published_time, "| url.host:", item.url.host)
print(item.model_dump())
print(item.model_dump_json())

`"14:19"` став справжнім `datetime.time(14, 19)` — з ним можна порівнювати й рахувати. `model_dump()` дає словник з об'єктами Python (`HttpUrl`, `time`), `model_dump_json()` — рядок JSON, де час уже `"14:19:00"`, а URL — текст.

### Приклад 2.3. Зіпсовані дані: усі помилки разом

In [ ]:
broken: RawNews = {"title": "Коротко", "url": "https://example.com/ukr/news/x.html",
                   "category": "", "description": "", "datetime": "25:99"}
try:
    NewsItem.from_raw(broken)
except ValidationError as error:
    bad_fields = {e["loc"][0] for e in error.errors()}
    print(error.error_count(), "помилки:")
    for e in error.errors():
        print(f"  {e['loc'][0]:15} {e['type']:20} {e['msg']}")

assert bad_fields == {"title", "url", "published_time"}, "одна ValidationError збирає всі помилки"

Одна `ValidationError` — **три** помилки: Pydantic не зупиняється на першій. `type` — машинний код (`string_too_short`, `value_error`, `time_parsing`), `msg` — пояснення. Для клієнта API це готовий список «що виправити» (FastAPI поверне його як `422`, урок 38).

### Приклад 2.4. `frozen=True` і пастка `model_copy`

In [ ]:
try:
    item.title = "Інший заголовок новини"
except ValidationError as error:
    print("присвоєння →", error.errors()[0]["msg"])

copy = item.model_copy(update={"title": "Ой"})
print("model_copy без перевірки →", repr(copy.title))
try:
    NewsItem.model_validate({**item.model_dump(), "title": "Ой"})
except ValidationError as error:
    print("model_validate →", error.errors()[0]["msg"])

Перевірену модель не змінити присвоєнням. `model_copy(update=...)` створює **нову** модель, але **не перевіряє** `update` — заголовок «Ой» коротший за 10 символів, а модель створилась. Якщо зміни приходять ззовні, перевіряй їх заново: `model_validate({**item.model_dump(), ...})`.

### Приклад 2.5. Приведення типів і пробіли

`str_strip_whitespace=True` обрізає пробіли в усіх рядках, а поле `time` приймає кілька форматів. Котрі саме?

In [ ]:
URL = "https://www.rbc.ua/ukr/news/budget-1.html"
for title, when in [("  Уряд ухвалив бюджет  ", "09:05"), ("Уряд ухвалив бюджет", "14:19:30"),
                    ("Уряд ухвалив бюджет", "9:05"), ("Уряд ухвалив бюджет", "2026-11-17T14:19")]:
    try:
        news = NewsItem.model_validate({"title": title, "url": URL, "published_time": when})
        print(f"{when!r:20} → {news.published_time}   title={news.title!r}")
    except ValidationError as error:
        print(f"{when!r:20} → помилка: {error.errors()[0]['msg']}")

- пробіли по краях заголовка зникли ще до перевірки довжини;
- `"09:05"` і `"14:19:30"` — правильний час ISO; `"9:05"` — **ні**: годину треба писати двома цифрами;
- повна дата-час — теж помилка: поле — `time`, а не `datetime`. Тип поля — це контракт: що не відповідає — не проходить.

### Приклад 2.6. Дані з URL: `model_validator(mode="before")`

HTML не містить ні мови, ні категорії — але вони є в URL. `derive_from_url` отримує **сирий** словник до перевірки полів і доповнює його: `urlsplit` ділить URL на домен і шлях, перша частина шляху → мова, друга → категорія.

In [ ]:
for url in ["https://www.rbc.ua/ukr/news/budget-1.html",
            "https://auto.rbc.ua/rus/news/speka-1778085857.html",
            "https://www.rbc.ua/ukr/economic/hryvnia-2.html",
            "https://www.rbc.ua/eng/news/budget-1.html"]:
    try:
        news = NewsItem.model_validate({"title": "Заголовок для перевірки", "url": url})
        print(news.source, news.lang, news.category)
    except ValidationError as error:
        print(url.split("/")[3], "→", [(e["loc"][0], e["msg"]) for e in error.errors()])

Піддомен `auto.rbc.ua` ні на що не вплинув: домен і шлях розібрано окремо. Для `/eng/` мову не виведено — і поле `lang` обов'язкове, тож помилка `Field required`. `mode="before"` — бо на цьому етапі полів `source`, `lang`, `category` ще немає: після перевірки полів (`mode="after"`) модель без них просто не створилася б.

### Приклад 2.7. Нічого не губиться мовчки: `validate_news`

`validate_news` розділяє сирі новини на **перевірені** й **відхилені** з причинами — відхилене можна показати, записати в журнал чи порахувати.

In [ ]:
valid_part, rejected_part = validate_news(raw_news[:2] + [broken])
print(len(valid_part), "перевірені:", [news.title[:25] for news in valid_part])
print(len(rejected_part), "відхилена:", type(rejected_part[0]).__name__)
for message in rejected_part[0].errors:
    print("  ", message)
print("сирі дані збережено:", rejected_part[0].raw["url"])

`Rejected` — теж Pydantic-модель: сирий словник (`raw`) і список причин. Парсер нічого не викидає сам, перевіряє лише модель — тому відхилену новину можна розібрати й виправити парсер.

---
## 3. Власні моделі

| Інструмент | Для чого |
|---|---|
| анотація поля | тип і приведення |
| `Field(min_length=, max_length=, ge=, le=, default_factory=)` | обмеження й значення за замовчуванням |
| `Literal[...]` | лише дозволені значення |
| `field_validator("поле")` | власне правило для одного поля |
| `model_validator(mode="before" / "after")` | обробка сирих даних / правило для кількох полів |
| `computed_field` | поле, яке обчислюється з інших і потрапляє в `model_dump()` |
| `model_config` | налаштування: `frozen`, `strict`, `extra`, `str_strip_whitespace` |

### Приклад 3.1. `Headline`: `Field` і `Literal`

Модель заголовка для майбутнього дайджесту: `title` — 10–300 символів, `lang` — лише `"uk"` або `"ru"`, `hour` — 0–23 або `None`.

In [ ]:
class Headline(BaseModel):
    title: str = Field(min_length=10, max_length=300)
    lang: Literal["uk", "ru"]
    hour: int | None = Field(default=None, ge=0, le=23)

In [ ]:
assert Headline(title="Уряд затвердив бюджет", lang="uk").hour is None
assert Headline(title="Уряд затвердив бюджет", lang="ru", hour="14").hour == 14, "рядок '14' → число"
print(Headline(title="Уряд затвердив бюджет", lang="ru", hour="14"))
for bad in ({"title": "Коротко", "lang": "uk"}, {"title": "Уряд затвердив бюджет", "lang": "en"},
            {"title": "Уряд затвердив бюджет", "lang": "uk", "hour": 25}):
    try:
        Headline(**bad)
        raise AssertionError(f"мало не пройти: {bad}")
    except ValidationError as error:
        print(bad, "→", error.errors()[0]["msg"])

`hour="14"` пройшов і став `14`: за замовчуванням Pydantic працює в «м'якому» режимі й приводить очевидні випадки (приклад 3.6 — суворий режим). `Literal` перелічив у помилці дозволені значення — `'uk' or 'ru'`.

### Приклад 3.2. `field_validator`: правило для одного поля

Заголовок з одних великих літер — клікбейт. Такого обмеження в `Field` немає — пишемо своє. Валідатор отримує **уже перевірене** значення (рядок ≥ 10 символів), повертає його або кидає `ValueError`.

In [ ]:
class CalmHeadline(BaseModel):
    title: str = Field(min_length=10)

    @field_validator("title")
    @classmethod
    def not_shouting(cls, title: str) -> str:
        if title.isupper():
            raise ValueError("заголовок-клікбейт")
        return title

In [ ]:
assert CalmHeadline(title="Уряд затвердив бюджет").title == "Уряд затвердив бюджет"
try:
    CalmHeadline(title="ТЕРМІНОВО!!! ШОК!!!")
    raise AssertionError("мало не пройти")
except ValidationError as error:
    print(error.errors()[0]["type"], "→", error.errors()[0]["msg"])

`ValueError` з валідатора Pydantic загорнув у звичайну помилку поля з типом `value_error` і префіксом `Value error,` — так само, як `only_rbc` у `NewsItem`.

### Приклад 3.3. `model_validator(mode="after")`: правило для кількох полів

Вікно дайджесту «з 08:00 до 12:00»: кінець має бути пізніше за початок. Правило стосується **двох** полів — перевіряємо готову модель (`mode="after"`), тут обидва поля вже мають тип `time`.

In [ ]:
class TimeWindow(BaseModel):
    start: time
    end: time

    @model_validator(mode="after")
    def end_after_start(self) -> "TimeWindow":
        if self.end <= self.start:
            raise ValueError("кінець вікна має бути пізніше за початок")
        return self

    def contains(self, moment: time | None) -> bool:
        return moment is not None and self.start <= moment < self.end

In [ ]:
morning = TimeWindow(start="08:00", end="12:00")
print(morning, "| новин у вікні:", sum(morning.contains(NewsItem.from_raw(raw).published_time) for raw in raw_news))
try:
    TimeWindow(start="18:00", end="09:00")
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])

У помилки з `mode="after"` `loc` порожній — `()`: вона стосується моделі в цілому, а не одного поля. Модель може мати й звичайні методи — `contains` використовує вже перевірені поля.

### Приклад 3.4. Вкладені моделі й шлях до помилки

Поле моделі може бути іншою моделлю або списком моделей. Pydantic перевірить кожен елемент — і скаже **де саме** помилка.

In [ ]:
class Digest(BaseModel):
    title: str
    created_at: datetime = Field(default_factory=datetime.now)
    items: list[NewsItem] = Field(default_factory=list)

In [ ]:
digest = Digest(title="Ранок", items=[{"title": raw["title"], "url": raw["url"]} for raw in raw_news[1:3]])
print(type(digest.items[0]).__name__, len(digest.items), "| created_at:", type(digest.created_at).__name__)
print("порожній дайджест:", Digest(title="Порожній").items)
try:
    Digest(title="Ранок", items=[{"title": raw_news[1]["title"], "url": raw_news[1]["url"]},
                                 {"title": "Коротко", "url": raw_news[2]["url"]}])
except ValidationError as error:
    print([(e["loc"], e["msg"]) for e in error.errors()])

- словники зі списку стали об'єктами `NewsItem` — з усіма їхніми перевірками й виведеними з URL полями;
- `loc = ('items', 1, 'title')` — «поле `items`, елемент з індексом 1, поле `title`»: точна адреса помилки у вкладених даних;
- `default_factory=list` створює **новий** список для кожного дайджесту, `default_factory=datetime.now` — свій час створення.

### Приклад 3.5. `computed_field`: обчислене поле

Іноді потрібне поле, якого немає у вхідних даних, — наприклад, кількість новин за мовами. `@computed_field` над `@property` додає його в `model_dump()` і в JSON Schema.

In [ ]:
class Feed(BaseModel):
    items: list[NewsItem]

    @computed_field
    @property
    def by_lang(self) -> dict[str, int]:
        return dict(Counter(item.lang for item in self.items))

In [ ]:
feed = Feed(items=[NewsItem.from_raw(raw) for raw in raw_news[:20]])
print("властивість:", feed.by_lang)
print("у model_dump:", {key: value for key, value in feed.model_dump().items() if key != "items"})
print("у схемі:", Feed.model_json_schema(mode="serialization")["properties"]["by_lang"])

`by_lang` не можна передати при створенні — його рахує модель. Схема для **виходу** (`mode="serialization"`) знає про поле і позначає його `readOnly`: клієнт API бачить, що його не треба надсилати.

### Приклад 3.6. Суворий режим: `strict=True`

«М'який» режим приводить `"14"` до `14`. Для внутрішніх даних це зручно, але іноді потрібно, щоб тип був **точно** таким.

In [ ]:
class LaxHour(BaseModel):
    hour: int


class StrictHour(BaseModel):
    model_config = ConfigDict(strict=True)
    hour: int

In [ ]:
for model in (LaxHour, StrictHour):
    for value in (14, "14", 14.0, True):
        try:
            print(f"{model.__name__:10} {value!r:6} → {model(hour=value).hour!r}")
        except ValidationError as error:
            print(f"{model.__name__:10} {value!r:6} → {error.errors()[0]['msg']}")

М'який режим прийняв рядок `"14"`, `14.0` (дробової частини немає — нічого не губиться) і навіть `True` → `1`: у Python `bool` — підклас `int`. Суворий режим приймає лише справжній `int`, і `True` теж не пройшов. Для даних з HTML і JSON-запитів зазвичай лишають м'який режим: там усе приходить рядками.

### Приклад 3.7. Зайві поля: ігнорувати чи відхиляти

За замовчуванням Pydantic мовчки **ігнорує** невідомі поля. Для вхідних даних API краще знати про одруківку: `extra="forbid"`.

In [ ]:
class SubscribeIn(BaseModel):
    chat_id: int
    keyword: str = Field(min_length=2)


class StrictSubscribeIn(SubscribeIn):
    model_config = ConfigDict(extra="forbid")

In [ ]:
data = {"chat_id": 42, "keywrod": "бюджет", "keyword": "економіка"}
print("ignore →", SubscribeIn(**data))
try:
    StrictSubscribeIn(**data)
except ValidationError as error:
    print("forbid →", [(e["loc"], e["msg"]) for e in error.errors()])

Одруківка `keywrod` у першій моделі просто зникла — клієнт не дізнається, що поле не дійшло. З `extra="forbid"` — помилка з назвою зайвого поля. Модель-нащадок успадкувала поля й додала лише налаштування.

### Приклад 3.8. `model_dump`: що віддавати назовні

In [ ]:
print(item.model_dump(include={"title", "lang"}))
print(item.model_dump(exclude={"url", "title"}))
print(item.model_dump(mode="json", exclude={"title"}))

`include` / `exclude` обирають поля, `mode="json"` перетворює значення на JSON-сумісні (`time` → рядок, `HttpUrl` → рядок), але повертає словник, а не рядок. Так готують дані для бази, кешу чи відповіді API.

### Задача 2. Звіт парсингу `ScrapeReport`

Опиши модель `ScrapeReport` — відповідь майбутнього `POST /api/scrape` (урок 38):

- `source: HttpUrl` — яку сторінку парсили;
- `scraped_at: datetime` — коли; за замовчуванням — зараз (`Field(default_factory=datetime.now)`);
- `items: list[NewsItem]`, `rejected: list[Rejected]`;
- `@computed_field` + `@property` `total -> int` — скільки всього новин (перевірених + відхилених).

Перевірка — у наступній клітинці.

In [ ]:
class ScrapeReport(BaseModel):
    # BEGIN SOLUTION
    source: HttpUrl
    scraped_at: datetime = Field(default_factory=datetime.now)
    items: list[NewsItem]
    rejected: list[Rejected]

    @computed_field
    @property
    def total(self) -> int:
        return len(self.items) + len(self.rejected)
    # END SOLUTION

In [ ]:
spoiled: RawNews = {"title": "Коротко", "url": "https://example.com/ukr/news/x.html",
                    "category": "", "description": "", "datetime": ""}
valid_part, rejected_part = validate_news(raw_news[:3] + [spoiled])
report = ScrapeReport(source="https://www.rbc.ua/ukr/news", items=valid_part, rejected=rejected_part)
dumped = report.model_dump()
assert "total" in dumped, "total має бути в model_dump(): @computed_field над @property"
assert dumped["total"] == 4 and len(dumped["items"]) == 3 and len(dumped["rejected"]) == 1
assert isinstance(report.source, HttpUrl), "source — HttpUrl, а не рядок"
assert isinstance(report.scraped_at, datetime) and abs((datetime.now() - report.scraped_at).total_seconds()) < 60
later = ScrapeReport(source="https://www.rbc.ua/", items=[], rejected=[])
assert later.scraped_at >= report.scraped_at, "default_factory: час — свій для кожного звіту"
try:
    ScrapeReport(source="не адреса", items=[], rejected=[])
    raise AssertionError("неправильний source має дати ValidationError")
except ValidationError:
    pass
print(report.model_dump_json(include={"source", "total"}))
print("✅ Задача 2 розв'язана")

---
## 4. Весь знімок, JSON і JSON Schema

### Приклад 4.1. Перевіряємо весь знімок

In [ ]:
valid, rejected = validate_news(raw_news)
n_valid = len(valid)
n_no_time = sum(item.published_time is None for item in valid)

assert (n_valid, len(rejected), n_no_time) == (168, 0, 31)
print("перевірених:", n_valid, "| відхилених:", len(rejected), "| без часу:", n_no_time)
print(Counter((n.source, n.lang, n.category) for n in valid))
print(count_by_lang(valid))

Увесь знімок пройшов: стартовий парсер збирав коректні новини. Але тепер це **доведено** кодом (і тестом `test_snapshot_is_valid`), а не припущено. Далі в агрегаторі живуть лише `NewsItem` — з гарантіями: `url` — справжній URL з rbc.ua, `lang` — `"uk"` або `"ru"`, `published_time` — `time` або `None`.

**🔮 Прогноз:** Яким буде `published_time` у `model_dump_json()` — `14:19` чи `14:19:00`? І чи можна з цього JSON отримати ту саму модель?

<details>
<summary>Відповідь</summary>

`"14:19:00"`: у моделі це `datetime.time`, а JSON-формат часу — ISO з секундами. З JSON модель відновлюється `model_validate_json` — і дорівнює початковій.

</details>

### Приклад 4.2. JSON туди й назад

In [ ]:
text = valid[1].model_dump_json()
print(text)
restored = NewsItem.model_validate_json(text)
print("та сама модель?", restored == valid[1], "| published_time:", repr(restored.published_time))

Моделі з однаковими полями рівні (`==`). `model_validate_json` не просто читає JSON, а **перевіряє** його тими самими правилами — так FastAPI прийматиме тіло запиту (урок 38), а кеш — віддаватиме збережене (урок 40).

### Приклад 4.3. JSON Schema — опис моделі для інших програм

In [ ]:
schema = NewsItem.model_json_schema()
required = schema["required"]

assert required == ["title", "url", "source", "lang", "category"], "published_time має значення за замовчуванням"
print(required)
for field in ("title", "url", "lang", "published_time"):
    print(f"{field:15}", schema["properties"][field])

Обмеження моделі стали машинним описом: `minLength`/`maxLength`, `format: uri`, `enum`, «або час, або `null`». FastAPI вставить цю схему в `/docs`, а клієнт іншою мовою може згенерувати з неї свої класи.

### Приклад 4.4. `TypeAdapter`: перевірка без моделі

Не все треба загортати в клас. `TypeAdapter(тип)` перевіряє й приводить значення будь-якого типу — `list[int]`, `dict[str, time]`, `TypedDict`…

In [ ]:
adapter = TypeAdapter(list[int])
numbers = adapter.validate_python(["1", "2", "3"])

assert numbers == [1, 2, 3]
print(numbers)
try:
    adapter.validate_python(["1", "два"])
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])

schedule = TypeAdapter(dict[str, time]).validate_python({"ранок": "08:00", "вечір": "20:30"})
print(schedule)

`loc = (1,)` — елемент з індексом 1. Словник «назва → час» перевірено й приведено так само, як поле моделі.

### Приклад 4.5. Перевірений знімок: `TypeAdapter(list[RawNews])`

Так `snapshot.py` перевіряє файл знімка, перш ніж віддати його далі: `TypeAdapter(list[RawNews])`. Підсунемо йому зіпсований файл — у новини немає ключа `url`, а `title` — число.

In [ ]:
bad_file = Path(tempfile.mkdtemp()) / "bad_snapshot.json"
bad_file.write_text(json.dumps([raw_news[0], {"title": 42, "category": "", "description": "", "datetime": ""}],
                               ensure_ascii=False), encoding="utf-8")
try:
    load_snapshot(bad_file)
except ValidationError as error:
    for e in error.errors():
        print(e["loc"], e["msg"])
print("справжній знімок:", len(load_snapshot()), "новин")

`TypedDict` для mypy — лише підказка, а з `TypeAdapter` він став перевіркою: `(1, 'title')` — не рядок, `(1, 'url')` — немає ключа. Зіпсований файл не потрапить далі у вигляді «майже правильних» словників.

### Приклад 4.6. Список моделей у файл і назад

In [ ]:
news_list = TypeAdapter(list[NewsItem])
out_file = Path(tempfile.mkdtemp()) / "validated.json"
out_file.write_bytes(news_list.dump_json(valid[:3], indent=1))
print(out_file.read_text(encoding="utf-8")[:260], "…")
back = news_list.validate_json(out_file.read_bytes())
print("прочитано:", len(back), type(back[0]).__name__, "| однакові:", back == valid[:3])

`dump_json` дає байти JSON для всього списку, `validate_json` читає їх і знову перевіряє кожну новину. Так можна зберігати вже перевірені новини між запусками.

### Приклад 4.7. Від HTML до моделей: парсер + перевірка

Повний шлях агрегатора на крихітній «сторінці» без мережі. Парсер має дві стратегії: шукати контейнери новин за CSS-класами, а якщо їх немає — посилання `/news/` з текстом «14:19 Заголовок».

In [ ]:
HTML_CONTAINERS = """
<div class="newsline__item"><time>09:30</time>
  <a class="newsline__title" href="/ukr/news/uryad-shvalyv-byudzhet-1.html">Уряд схвалив проєкт бюджету</a></div>
<div class="newsline__item"><time>10:05</time>
  <a class="newsline__title" href="https://evil.example/ukr/news/fake.html">Фейкова новина з чужого сайту</a></div>
"""

HTML_LINKS = """
<a href="/ukr/news/">Новини</a>
<a href="/rus/news/kurs-dolara-2.html">11:40 Курс долара на міжбанку зріс</a>
<a href="/rus/news/kurs-dolara-2.html">11:40 Курс долара на міжбанку зріс</a>
<a href="/ukr/economic/infliatsiia-3.html">12:15 Інфляція сповільнилася</a>
"""

In [ ]:
for name, html in (("контейнери", HTML_CONTAINERS), ("посилання", HTML_LINKS)):
    raw = parse_rbc_news(html)
    ok, bad = validate_news(raw)
    print(f"{name}: сирих {len(raw)} → перевірених {len(ok)}, відхилених {len(bad)}")
    for news in ok:
        print("   ", news.published_time, news.lang, news.category, "|", news.title)
    for rejected_item in bad:
        print("    відхилено:", rejected_item.raw["url"], rejected_item.errors)

- контейнери: дві сирі новини, але «новина» з чужого домену відхилена валідатором `only_rbc` — з причиною;
- посилання: з чотирьох лишилось одне. Пункт меню «Новини» коротший за 10 символів, дубль відсіяла множина `seen_urls`, а «Інфляція» має адресу `/ukr/economic/…` — друга стратегія бере лише посилання з `/news/`. Час `11:40` парсер вирізав з початку тексту посилання;
- парсер лише збирає рядки, а все, що стосується правильності даних, вирішує модель.

---
## 5. Знайди помилку

Так прототип `news_dashboard` визначав розділ сайту:

```python
url_parts = [p for p in url.replace("https://www.rbc.ua", "").split("/") if p]
raw_cat = url_parts[1]
```

Для новини з `https://auto.rbc.ua/rus/news/…` «розділом» стає `auto.rbc.ua`.

### Приклад 5.1. Відтворюємо помилку на справжньому URL зі знімка

In [ ]:
CATEGORY_MAP = {"news": "Новини", "economic": "Економіка", "politics": "Політика"}


def category_by_replace(url):
    url_parts = [p for p in url.replace("https://www.rbc.ua", "").split("/") if p]
    raw_cat = url_parts[1] if len(url_parts) > 1 else (url_parts[0] if url_parts else "")
    return CATEGORY_MAP.get(raw_cat, raw_cat.capitalize() or "Загальні")

In [ ]:
auto_url = next(raw["url"] for raw in raw_news if "auto.rbc.ua" in raw["url"])
for url in (raw_news[1]["url"], auto_url, "http://www.rbc.ua/ukr/news/x.html"):
    print(f"{url[:45]:47} → {category_by_replace(url)}")

Для `www.rbc.ua` — «Новини», для `auto.rbc.ua` — «Auto.rbc.ua», а для `http://` (без `s`) — «Www.rbc.ua». `replace` прибирає префікс лише при **точному** збігу рядка: для іншого домену чи протоколу нічого не прибрано, `split("/")` дає `['https:', 'auto.rbc.ua', 'rus', …]`, і «категорією» стає домен. Помилка тиха — жодного винятку, лише неправильні дані в дашборді.

### Задача 3. Розділ з шляху URL

Напиши `section_of(url)` через `urllib.parse.urlsplit`: друга частина **шляху** URL (`/rus/news/…` → `"news"`), незалежно від домену й протоколу. Якщо в шляху менше двох частин — порожній рядок.

Перевірка — у наступній клітинці.

In [ ]:
def section_of(url: str) -> str:
    # BEGIN SOLUTION
    path = [segment for segment in urlsplit(url).path.split("/") if segment]
    return path[1] if len(path) > 1 else ""
    # END SOLUTION

In [ ]:
assert section_of(auto_url) == "news", f"піддомен не має впливати: {auto_url} → {section_of(auto_url)!r}"
assert section_of("https://www.rbc.ua/ukr/economic/hryvnia-2.html") == "economic"
assert section_of("http://www.rbc.ua/ukr/news/x.html") == "news", "протокол теж не має впливати"
assert section_of("https://www.rbc.ua/ukr/news/x.html?utm=tg#top") == "news", "параметри запиту — не шлях"
assert section_of("https://www.rbc.ua/") == ""
assert section_of("https://www.rbc.ua/ukr") == ""
print(auto_url, "→", section_of(auto_url))
print("✅ Задача 3 розв'язана")

---
## 6. Тести і `mypy --strict`

Рефакторинг закріплюють тести (`tests/test_parser.py`, `tests/test_models.py`) і суворий mypy на всьому пакеті. Ті самі команди ти запускатимеш у терміналі в папці `news_hub`; CI (урок 51) запускає їх на кожен push.

### Приклад 6.1. `pytest` і `mypy --strict news_hub`

In [ ]:
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1])
strict = run_mypy("--strict", "news_hub")
assert tests.returncode == 0 and strict.returncode == 0

Усі тести зелені, а `--strict` не знайшов жодної проблеми: кожна функція пакета має анотації, неявних `Any` немає. Тепер змінимо модель — і mypy та тести покажуть усі місця, які це зачепило.

---
## Самоперевірка

1. Що перевіряє `mypy`, а що — Pydantic? Чи можна обійтися чимось одним?
2. Чим `TypedDict` кращий за `dict` для результату парсера?
3. Навіщо `published_time: time | None`?
4. Що дає `frozen=True` і чому `model_copy(update=...)` — не перевірка?
5. Коли `model_validator(mode="before")`, а коли `mode="after"`?
6. Навіщо `extra="forbid"` для вхідних даних API?
7. Чому URL розбирають `urlsplit`, а не `replace`/`split`?

<details>
<summary>Відповіді</summary>

1. `mypy` — код до запуску (типи, ключі, `None`); Pydantic — дані під час роботи. Потрібні обидва: бездоганно типізований код може отримати з інтернету будь-які дані.
2. mypy знає ключі: одруківка — помилка до запуску, а не `KeyError` у роботі.
3. У 31 новини часу немає; тип чесно це описує, і mypy змусить обробити `None`.
4. Перевірену модель не можна змінити присвоєнням. `model_copy` створює нову модель без перевірки `update`; для даних ззовні — `model_validate`.
5. `before` — обробити сирі дані до перевірки полів (вивести `lang` з URL); `after` — правило для кількох уже перевірених полів (кінець пізніше за початок).
6. Щоб одруківка в назві поля стала помилкою, а не мовчки зникла.
7. `urlsplit` розбирає URL за стандартом (домен, шлях, параметри окремо); `replace` працює лише для точного збігу рядка.

</details>

## Далі

- Книга курсу: [Урок 37](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_37/) — рефакторинги «типи» і «Pydantic», архітектура межі довіри.
- У папці `news_hub`: `pytest` і `mypy --strict news_hub`.
- **Урок 38** — FastAPI над `NewsItem`: `/api/news`, `/docs`, Postman.